In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
!pip install -U transformers accelerate

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import argparse
import json
import os
import random
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

# -----------------------
# helpers
# -----------------------
def load_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        # pick first list value
        for v in data.values():
            if isinstance(v, list):
                return v
    raise ValueError(f"Unsupported JSON format in {path}. Expected list or dict containing a list.")

def pick_first_existing(d: Dict[str, Any], keys: List[str]) -> Optional[str]:
    for k in keys:
        if k in d and d[k] is not None:
            return k
    return None

def normalize_label(x: Any) -> Any:
    if isinstance(x, str):
        return x.strip().lower()
    return x

def build_label_maps(records: List[Dict[str, Any]], label_key: str,
                     explicit_labels: Optional[List[str]] = None) -> Tuple[Dict[Any, int], Dict[int, str]]:
    # If explicit labels provided, use those in given order
    if explicit_labels:
        labels_norm = [l.strip().lower() for l in explicit_labels]
        label2id = {l: i for i, l in enumerate(labels_norm)}
        id2label = {i: l for l, i in label2id.items()}
        return label2id, id2label

    # Infer label space from data
    uniq = sorted({normalize_label(r[label_key]) for r in records})
    # If they are strings, keep them. If ints, map ints to themselves.
    if all(isinstance(u, (int, np.integer)) for u in uniq):
        label2id = {int(u): i for i, u in enumerate(uniq)}
        id2label = {i: str(u) for u, i in label2id.items()}
        return label2id, id2label

    # strings
    uniq = [str(u) for u in uniq]
    label2id = {u: i for i, u in enumerate(uniq)}
    id2label = {i: u for u, i in label2id.items()}
    return label2id, id2label

def encode_batch(examples: Dict[str, List[Any]], tokenizer, premise_key: str, hypothesis_key: str, max_length: int):
    return tokenizer(
        examples[premise_key],
        examples[hypothesis_key],
        truncation=True,
        max_length=max_length,
    )

def compute_metrics_builder(id2label: Dict[int, str]):
    labels_sorted = [id2label[i] for i in range(len(id2label))]

    def compute_metrics(eval_pred):
        logits, y_true = eval_pred
        y_pred = np.argmax(logits, axis=-1)

        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

        cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels_sorted))))

        # pack per-class f1 into dict
        per_class = {labels_sorted[i]: float(per_class_f1[i]) for i in range(len(labels_sorted))}

        return {
            "accuracy": float(acc),
            "macro_f1": float(macro_f1),
            **{f"f1_{k}": v for k, v in per_class.items()},
            # confusion matrix cannot be a numpy array for trainer logging
            "confusion_matrix": cm.tolist(),
        }

    return compute_metrics


In [ ]:
import json
import random
from pathlib import Path

# -----------------------
# config
# -----------------------
INPUT_JSON = "/kaggle/input/data4good/train.json"
TRAIN_OUT = "train_90.json"
VAL_OUT = "val_10.json"
VAL_RATIO = 0.10
SEED = 42

# -----------------------
# load data
# -----------------------
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

# handle dict-wrapped datasets
if isinstance(data, dict):
    # assume first list-like value is the dataset
    for v in data.values():
        if isinstance(v, list):
            records = v
            break
    else:
        raise ValueError("No list found in JSON")
else:
    records = data

n = len(records)
assert n > 0, "Empty dataset"

# -----------------------
# shuffle + split
# -----------------------
random.seed(SEED)
random.shuffle(records)

val_size = int(n * VAL_RATIO)
val_data = records[:val_size]
train_data = records[val_size:]

print(f"Total samples: {n}")
print(f"Train: {len(train_data)}")
print(f"Val: {len(val_data)}")

# -----------------------
# save
# -----------------------
with open(TRAIN_OUT, "w", encoding="utf-8") as f:
    json.dump(train_data, f, indent=2)

with open(VAL_OUT, "w", encoding="utf-8") as f:
    json.dump(val_data, f, indent=2)


In [ ]:
train_json = '/kaggle/working/train_90.json'
val_json = '/kaggle/working/val_10.json'

model = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli'
out_dir = 'ft_nli_out'

premise_key = 'context'
hypothesis_key = 'answer'
label_key = 'type'

labels = ['factual', 'contradiction', 'irrelevant']

max_length = 256

lr = 2e-5

batch_size = 8
epochs  = 5
weight_decay = 0.01
seed = 42

fp16 = False
bf16 = False

set_seed(seed)

train_records = load_json_list(train_json)
val_records = load_json_list(val_json)

# auto-detect keys if not provided
sample = train_records[0]
premise_key = premise_key or pick_first_existing(sample, ["context", "premise", "passage", "input", "text", "question"])
hypothesis_key = hypothesis_key or pick_first_existing(sample, ["answer", "hypothesis", "response", "candidate", "completion"])
label_key = label_key or pick_first_existing(sample, ["label", "gold", "y", "target", "class"])

if premise_key is None or hypothesis_key is None or label_key is None:
    raise ValueError(
        f"Could not auto-detect keys. Found premise={premise_key}, hypothesis={hypothesis_key}, label={label_key}. "
        f"Pass --premise_key, --hypothesis_key, --label_key explicitly."
    )
# build label maps from TRAIN ONLY
label2id, id2label = build_label_maps(train_records, label_key, explicit_labels=labels)

def remap_labels(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    for r in records:
        rr = dict(r)
        y = normalize_label(rr[label_key])
        if labels:
            y = str(y).strip().lower()
        if y not in label2id:
            raise ValueError(f"Label '{y}' not in label2id {label2id}. Fix your label set or pass --labels.")
        rr["labels"] = int(label2id[y])
        out.append(rr)
    return out

train_records = remap_labels(train_records)
val_records = remap_labels(val_records)

train_ds = Dataset.from_list(train_records)
val_ds = Dataset.from_list(val_records)

tokenizer = AutoTokenizer.from_pretrained(model, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(
    model,
    num_labels=len(label2id),
    id2label={i: id2label[i] for i in id2label},
    label2id={k: v for k, v in label2id.items()},
)

train_ds = train_ds.map(
    lambda ex: encode_batch(ex, tokenizer, premise_key, hypothesis_key, max_length),
    batched=True,
    remove_columns=[c for c in train_ds.column_names if c not in (premise_key, hypothesis_key, "labels")]
)
val_ds = val_ds.map(
    lambda ex: encode_batch(ex, tokenizer, premise_key, hypothesis_key, max_length),
    batched=True,
    remove_columns=[c for c in val_ds.column_names if c not in (premise_key, hypothesis_key, "labels")]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=out_dir,
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    weight_decay=weight_decay,

    do_eval=True,
    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_steps=50,
    report_to=[],
    fp16=fp16,
    bf16=bf16,
    seed=seed,
)


In [ ]:
# trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_builder(id2label),
)

trainer.train()

# final eval with extra reporting
preds = trainer.predict(val_ds)
logits = preds.predictions
y_true = preds.label_ids
y_pred = np.argmax(logits, axis=-1)

labels_sorted = [id2label[i] for i in range(len(id2label))]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels_sorted))))

print("\n===== FINAL VALIDATION METRICS =====")
print(f"Accuracy:  {acc:.4f}")
print(f"Macro F1:  {macro_f1:.4f}")
print("\nPer-class F1:")
for i, name in enumerate(labels_sorted):
    print(f"  {name}: {per_class_f1[i]:.4f}")

print("\nConfusion matrix (rows=true, cols=pred), label order:")
print(labels_sorted)
print(cm)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=labels_sorted, digits=4, zero_division=0))

os.makedirs(out_dir, exist_ok=True)
with open(os.path.join(out_dir, "val_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "accuracy": float(acc),
            "macro_f1": float(macro_f1),
            "per_class_f1": {labels_sorted[i]: float(per_class_f1[i]) for i in range(len(labels_sorted))},
            "confusion_matrix": cm.tolist(),
            "label_order": labels_sorted,
            "premise_key": premise_key,
            "hypothesis_key": hypothesis_key,
            "label_key": label_key,
            "model": model,
        },
        f,
        indent=2,
    )

print(f"\nSaved metrics to: {os.path.join(out_dir, 'val_metrics.json')}")
print(f"Saved checkpoints to: {out_dir}")


## prediction

In [2]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

ckpt_dir = "/kaggle/working/ft_nli_out/checkpoint-1183"

tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(ckpt_dir)

def preprocess(examples):
    # Same formatting rule: match training
    texts = [
        f"Question: {q}\nContext: {c}\nAnswer: {a}"
        for q, c, a in zip(examples["Question"], examples["Context"], examples["Answer"])
    ]
    return tokenizer(texts, truncation=True, padding=False, max_length=512)

def predict_probs(df: pd.DataFrame, batch_size=32):
    ds = Dataset.from_pandas(df)
    ds = ds.map(preprocess, batched=True, remove_columns=[col for col in ds.column_names if col not in ["Question","Context","Answer"]])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask"] + (["token_type_ids"] if "token_type_ids" in ds.features else []))

    args = TrainingArguments(
        output_dir="tmp_pred",
        per_device_eval_batch_size=batch_size,
        dataloader_drop_last=False,
        report_to=[],
    )

    trainer = Trainer(model=model, args=args, tokenizer=tokenizer)
    pred = trainer.predict(ds)

    logits = pred.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()
    pred_ids = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    return pred_ids, conf, probs



2025-12-22 00:05:19.180853: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766361919.348163      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766361919.391514      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766361919.774859      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766361919.774886      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766361919.774889      55 computation_placer.cc:177] computation placer alr

In [3]:
train_df = pd.read_csv("/kaggle/input/data4good/train.json")

In [4]:

train_pred_ids, train_conf, train_probs = predict_probs(train_df, batch_size=64)
# test_pred_ids,  test_conf,  test_probs  = predict_probs(test_df, batch_size=64)

train_out = train_df.copy()
train_out["pred_id"] = train_pred_ids
train_out["pred_conf"] = train_conf

# test_out = test_df.copy()
# test_out["pred_id"] = test_pred_ids
# test_out["pred_conf"] = test_conf

train_out.to_csv("train_preds.csv", index=False)
# test_out.to_csv("test_preds.csv", index=False)
print("Saved train_preds.csv and test_preds.csv")


KeyboardInterrupt: 

In [6]:
import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, TrainingArguments, Trainer

# -------------------------
# CONFIG
# -------------------------
CKPT_DIR = "/kaggle/working/ft_nli_out/checkpoint-1183"          # path to your checkpoint folder
TRAIN_JSON = "/kaggle/input/data4good/train.json"            # path to train json
TEST_JSON  = "/kaggle/input/data4good-test/test.json"             # path to test json
OUT_DIR = "pred_outputs"
BATCH_SIZE = 64
MAX_LEN = 512
USE_FP16 = True

os.makedirs(OUT_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available. You are not running on a GPU runtime.")
print("GPU:", torch.cuda.get_device_name(0))

# -------------------------
# LOAD MODEL + TOKENIZER + LABEL MAP
# -------------------------
cfg = AutoConfig.from_pretrained(CKPT_DIR)
tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR)

id2label = dict(cfg.id2label)
label2id = dict(cfg.label2id)
num_labels = cfg.num_labels

print("id2label from checkpoint:", id2label)

# If you see LABEL_0/LABEL_1/LABEL_2 here, set your real mapping.
# IMPORTANT: This mapping must match the label order used during training.
# Example (FIX ORDER if needed):
# manual_id2label = {0: "factual", 1: "contradiction", 2: "irrelevant"}
# manual_label2id = {v: k for k, v in manual_id2label.items()}
# if list(id2label.values())[0].startswith("LABEL_"):
#     id2label = manual_id2label
#     label2id = manual_label2id

# -------------------------
# READ JSON (robust)
# -------------------------
def read_json_any(path: str) -> pd.DataFrame:
    """
    Supports:
    1) JSON Lines (one object per line)
    2) A single JSON array of objects
    """
    try:
        df = pd.read_json(path, lines=True)
        if len(df) > 0:
            return df
    except ValueError:
        pass
    # fallback: normal JSON
    return pd.read_json(path)

train_df = read_json_any(TRAIN_JSON)
test_df  = read_json_any(TEST_JSON)

# -------------------------
# VALIDATE REQUIRED COLS
# -------------------------
required = {"question", "context", "answer"}
missing_train = required - set(train_df.columns)
missing_test  = required - set(test_df.columns)
if missing_train:
    raise ValueError(f"Train JSON missing columns: {missing_train}")
if missing_test:
    raise ValueError(f"Test JSON missing columns: {missing_test}")

# -------------------------
# TEXT BUILD (must match training format)
# -------------------------
def build_text(q, c, a):
    # Change this ONLY if training used a different format.
    return f"Question: {q}\nContext: {c}\nAnswer: {a}"

def preprocess(examples):
    texts = [
        build_text(q, c, a)
        for q, c, a in zip(examples["Question"], examples["Context"], examples["Answer"])
    ]
    return tokenizer(texts, truncation=True, max_length=MAX_LEN, padding=False)

# -------------------------
# PREDICT (GPU)
# -------------------------
def predict_gpu(df: pd.DataFrame) -> pd.DataFrame:
    base_cols = ["Question", "Context", "Answer"]
    has_labels = "Type" in df.columns

    df_in = df[base_cols + (["Type"] if has_labels else [])].copy()

    ds = Dataset.from_pandas(df_in)
    ds = ds.map(preprocess, batched=True)

    # If labels exist, convert to ids so we can keep true_label/true_id in output
    if has_labels:
        if pd.api.types.is_numeric_dtype(df_in["Type"]):
            labels = df_in["Type"].astype(int).tolist()
        else:
            if not label2id:
                raise ValueError("label2id is empty in config. Provide manual mapping.")
            missing = set(df_in["Type"].unique()) - set(label2id.keys())
            if missing:
                raise ValueError(f"Found labels not in label2id: {missing}. Fix label mapping.")
            labels = [label2id[x] for x in df_in["Type"].tolist()]
        ds = ds.add_column("labels", labels)

    cols = ["input_ids", "attention_mask"]
    if "token_type_ids" in ds.features:
        cols.append("token_type_ids")
    if "labels" in ds.features:
        cols.append("labels")
    ds.set_format(type="torch", columns=cols)

    args = TrainingArguments(
        output_dir=os.path.join(OUT_DIR, "tmp_pred"),
        per_device_eval_batch_size=BATCH_SIZE,
        fp16=USE_FP16,
        report_to=[],
    )
    trainer = Trainer(model=model, args=args, tokenizer=tokenizer)

    pred = trainer.predict(ds)
    logits = pred.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()
    pred_id = probs.argmax(axis=1)
    pred_conf = probs.max(axis=1)

    out = df_in[base_cols].copy()

    if has_labels:
        out["true_label"] = df_in["Type"].values
        if pd.api.types.is_numeric_dtype(df_in["Type"]):
            out["true_id"] = df_in["Type"].astype(int).values
        else:
            out["true_id"] = np.array([label2id[x] for x in df_in["Type"].tolist()], dtype=int)

    out["pred_id"] = pred_id
    out["pred_label"] = [id2label[int(i)] for i in pred_id]
    out["pred_conf"] = pred_conf

    for k in range(num_labels):
        out[f"prob_{id2label[k]}"] = probs[:, k]

    return out

train_out = predict_gpu(train_df)
test_out  = predict_gpu(test_df)

train_csv = os.path.join(OUT_DIR, "train_predictions_full.csv")
test_csv  = os.path.join(OUT_DIR, "test_predictions_full.csv")

train_out.to_csv(train_csv, index=False)
test_out.to_csv(test_csv, index=False)

print("Saved:", train_csv)
print("Saved:", test_csv)

# Optional submission (common format: id + Type)
# if "id" in test_df.columns:
#     sub = pd.DataFrame({"id": test_df["id"], "Type": test_out["pred_label"]})
#     sub.to_csv(os.path.join(OUT_DIR, "submission.csv"), index=False)
#     print("Saved submission.csv")


GPU: Tesla T4
id2label from checkpoint: {0: 'factual', 1: 'contradiction', 2: 'irrelevant'}


ValueError: Train JSON missing columns: {'Question', 'Answer', 'Context'}

In [10]:
train_df

,0,1,2,3,4,5,6,7,8,9,...,21011,21012,21013,21014,21015,21016,21017,21018,21019,21020
0,"{'answer': 'In 1512, Parliament passed a signi...",{'answer': 'The Spanish and French were the on...,"{'answer': 'Traditionally, monsoons in Punjab ...",{'answer': 'The media made the requests for Ko...,{'answer': 'According to historians Robert Fri...,{'answer': 'The process that can increase sola...,{'answer': 'The main focus of the 5th season i...,{'answer': 'The mean annual temperature in Hyd...,{'answer': 'The issue of through traffic benef...,{'answer': 'Montini's office received nearly t...,...,"{'answer': 'Jehovah's Witnesses use the term ""...",{'answer': 'The University of Paris is credite...,{'answer': 'The Soviets opposed the rebuilding...,{'answer': 'The Super Scope is a light gun acc...,"{'answer': 'In 1988, there were 60,000 compute...","{'answer': 'At the first proceeding, the jury ...",{'answer': 'King Henry VII of England commissi...,{'answer': 'The distinguishing visual design f...,{'answer': 'For group law and topology to inte...,{'answer': 'The lead boat on March 13 was PCF-...


In [17]:
import json
import pandas as pd

def to_records(obj):
    """
    Convert common JSON shapes into a list[dict] records.
    Handles:
      1) list[dict]
      2) dict keyed by "0","1",... where values are dict records
      3) dict wrapper where one value is list[dict] or dict-of-dicts
    """
    # Case 1: already list of dicts
    if isinstance(obj, list):
        if len(obj) == 0:
            return []
        if all(isinstance(x, dict) for x in obj):
            return obj
        raise ValueError("List JSON is not a list of dict records.")

    # Case 2: dict-of-dicts keyed by indices
    if isinstance(obj, dict):
        # If wrapper dict: find first list-like or dict-of-dicts value
        # First try: if values are dicts (records) and keys look like indices
        if len(obj) > 0 and all(isinstance(v, dict) for v in obj.values()):
            # sort by numeric key if possible
            try:
                items = sorted(obj.items(), key=lambda kv: int(kv[0]))
                return [v for _, v in items]
            except Exception:
                return list(obj.values())

        # Wrapper dict: search for dataset inside
        for v in obj.values():
            if isinstance(v, list) and (len(v) == 0 or isinstance(v[0], dict)):
                return v
            if isinstance(v, dict) and len(v) > 0 and all(isinstance(x, dict) for x in v.values()):
                try:
                    items = sorted(v.items(), key=lambda kv: int(kv[0]))
                    return [vv for _, vv in items]
                except Exception:
                    return list(v.values())

    raise ValueError("Unrecognized JSON format. Provide list[dict] or dict-of-dicts.")

def load_json_to_df(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    records = to_records(obj)
    df = pd.DataFrame(records)

    # Optional: normalize column names to what your training expects
    rename_map = {}
    if "Question" in df.columns and "question" not in df.columns:
        rename_map["Question"] = "question"
    if "Context" in df.columns and "context" not in df.columns:
        rename_map["Context"] = "context"
    if "Answer" in df.columns and "answer" not in df.columns:
        rename_map["Answer"] = "answer"
    if "Type" in df.columns and "type" not in df.columns:
        rename_map["Type"] = "type"

    if rename_map:
        df = df.rename(columns=rename_map)

    return df



In [21]:
import os
import json
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

# -------------------------
# CONFIG (EDIT PATHS)
# -------------------------
CKPT_DIR   = "/kaggle/working/ft_nli_out/checkpoint-1183"   # or "checkpoint-1183"
TRAIN_JSON = "/kaggle/input/data4good/train.json"                  # your full train json or train_90.json
TEST_JSON  = "/kaggle/input/data4good-test/test.json"
OUT_DIR    = "pred_outputs"

premise_key = "context"
hypothesis_key = "answer"
label_key = "type"

LABELS = ["factual", "contradiction", "irrelevant"]  # training order
MAX_LEN = 256                                       # match training
BATCH_SIZE = 64                                     # increase if VRAM allows
USE_FP16 = True                                     # inference speedup on most GPUs

os.makedirs(OUT_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available. Run on GPU.")
print("GPU:", torch.cuda.get_device_name(0))

# -------------------------
# JSON LOADER (list JSON or JSONL)
# -------------------------
def load_json_any(path: str):
    # Try JSONL first
    try:
        df = pd.read_json(path, lines=True)
        if len(df) > 0:
            return df
    except ValueError:
        pass
    # Fallback: standard JSON
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # If dict-wrapped, take first list value
    if isinstance(data, dict):
        for v in data.values():
            if isinstance(v, list):
                data = v
                break
    if not isinstance(data, list):
        raise ValueError("JSON must be a list of records or JSONL.")
    return pd.DataFrame(data)

# -------------------------
# LOAD MODEL + TOKENIZER
# -------------------------
cfg = AutoConfig.from_pretrained(CKPT_DIR)
tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR)

# Enforce label names/order used in training (avoid LABEL_0 mistakes)
id2label = {i: LABELS[i] for i in range(len(LABELS))}
label2id = {v: k for k, v in id2label.items()}
num_labels = len(LABELS)

# -------------------------
# PREPROCESS = PAIR TOKENIZATION (context, answer)
# -------------------------
def preprocess(examples):
    # Pair encoding matches NLI-style training
    return tokenizer(
        examples[premise_key],
        examples[hypothesis_key],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )

# -------------------------
# PREDICT (GPU)
# -------------------------

def predict_gpu(df: pd.DataFrame, *, is_train: bool) -> pd.DataFrame:
    # required fields
    for col in [premise_key, hypothesis_key]:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in input JSON.")

    keep_cols = []
    for col in ["question", "Question", premise_key, hypothesis_key]:
        if col in df.columns and col not in keep_cols:
            keep_cols.append(col)

    if is_train:
        if label_key not in df.columns:
            raise ValueError("Training data must contain labels.")
        keep_cols.append(label_key)

    df_in = df[keep_cols].copy()

    ds = Dataset.from_pandas(df_in)
    ds = ds.map(preprocess, batched=True)

    # add labels ONLY for train
    if is_train:
        y = df_in[label_key].astype(str).str.strip().str.lower().tolist()
        missing = set(y) - set(label2id.keys())
        if missing:
            raise ValueError(f"Invalid labels in train: {missing}")
        ds = ds.add_column("labels", [label2id[v] for v in y])

    cols = ["input_ids", "attention_mask"]
    if "token_type_ids" in ds.features:
        cols.append("token_type_ids")
    if is_train:
        cols.append("labels")
    ds.set_format(type="torch", columns=cols)

    args = TrainingArguments(
        output_dir=os.path.join(OUT_DIR, "tmp_pred"),
        per_device_eval_batch_size=BATCH_SIZE,
        fp16=USE_FP16,
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=args,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )

    pred = trainer.predict(ds)
    logits = pred.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()

    pred_id = probs.argmax(axis=1)
    pred_conf = probs.max(axis=1)

    out = df_in.copy()

    # normalize question column
    if "Question" in out.columns and "question" not in out.columns:
        out = out.rename(columns={"Question": "question"})

    if is_train:
        out["true_label"] = out[label_key].astype(str).str.strip().str.lower()

    out["pred_id"] = pred_id
    out["pred_label"] = [id2label[int(i)] for i in pred_id]
    out["pred_conf"] = pred_conf

    out["prob_factual"] = probs[:, label2id["factual"]]
    out["prob_contradiction"] = probs[:, label2id["contradiction"]]
    out["prob_irrelevant"] = probs[:, label2id["irrelevant"]]

    return out



# -------------------------
# RUN
# -------------------------
# # train_df = load_json_to_df(TRAIN_JSON)
# print(train_df.shape) 

# test_df  = load_json_any(TEST_JSON)

# train_out = predict_gpu(train_df, is_train=True)
test_out  = predict_gpu(test_df,  is_train=False)


train_path = os.path.join(OUT_DIR, "train_predictions_full.csv")
test_path  = os.path.join(OUT_DIR, "test_predictions_full.csv")

train_out.to_csv(train_path, index=False)
test_out.to_csv(test_path, index=False)

print("Saved:", train_path)
print("Saved:", test_path)


GPU: Tesla T4


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1914300810.py:135: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Saved: pred_outputs/train_predictions_full.csv
Saved: pred_outputs/test_predictions_full.csv


In [22]:
train_out.query("true_label!=pred_label")

,question,context,answer,type,true_label,pred_id,pred_label,pred_conf,prob_factual,prob_contradiction,prob_irrelevant
9,Fireworks are discharged on New Year's Eve fro...,,Montini's office received nearly ten million i...,irrelevant,irrelevant,0,factual,0.889930,0.889930,0.017283,0.092787
55,What position is the gymnast when in the air?,,"In the 1960s, the life expectancy in Egypt was...",irrelevant,irrelevant,0,factual,0.887763,0.887763,0.016634,0.095603
78,When did the British Isles area become separat...,,The British Isles area became separated from t...,contradiction,contradiction,0,factual,0.878059,0.878059,0.020744,0.101197
169,What were the Moors who converted to Catholici...,The Statistics Portugal (Portuguese: INE - Ins...,Moriscos were the Moors who did not convert to...,contradiction,contradiction,0,factual,0.824362,0.824362,0.155296,0.020343
212,de Bellaigne attributed the growth of Islamism...,,de Bellaigne attributed the growth of Islamism...,contradiction,contradiction,0,factual,0.858625,0.858625,0.024632,0.116743
...,...,...,...,...,...,...,...,...,...,...,...
20537,What event evolving animals are the Myanmar al...,,Myanmar is also accredited with being the firs...,contradiction,contradiction,0,factual,0.876450,0.876450,0.021229,0.102321
20552,In which decade did wrestling start becoming v...,,"Particularly since the 1980s, pro wrestling ev...",contradiction,contradiction,0,factual,0.881179,0.881179,0.020074,0.098747
20784,Who did Chopin send his Preludes to?,"On 3 December, Chopin complained about his bad...",Chopin sent his Preludes to Liszt.,contradiction,contradiction,0,factual,0.999890,0.999890,0.000090,0.000020
20785,What is the term for a pregnant slave?,"Due to the patriarchal nature of Arab society,...","The term for a pregnant slave is ""umm al-walad.""",contradiction,contradiction,0,factual,0.919520,0.919520,0.076731,0.003749


In [35]:
test_out.pred_label.value_counts() / test_out.shape[0]

pred_label
factual          0.8345
irrelevant       0.0840
contradiction    0.0815
Name: count, dtype: float64

In [36]:
train_out.type.value_counts() / train_out.shape[0]

type
factual          0.829218
contradiction    0.086485
irrelevant       0.084297
Name: count, dtype: float64

In [37]:
from sklearn.metrics import f1_score, accuracy_score

In [40]:
f1_score(train_out.true_label, train_out.pred_label, average = 'macro')

0.974963934314232

In [43]:
f1_score(train_out.true_label, train_out.pred_label, average = None)

array([0.97853359, 0.99358371, 0.9527745 ])

In [42]:
accuracy_score(train_out.true_label, train_out.pred_label)

0.989010989010989